In [1]:
# %%
# Use the QRF regressor to make two monthly prediction products per variable, per pixel, over the full grid:

### FIX: DECIDING THE NUMBER OF ENSEMBLES - 10 kept for trial run


#   1. Deterministic  -- the model's median read, one band per day
#   2. Stochastic     -- 100 realizations per day, Schaake-shuffled
#                        (Clark et al. 2004) against a historically kriged archive
#                          to restore cross-pixel / cross-variable
#                        rank coherence that independent per-pixel QRF
#                        sampling doesn't have on its own
#
# Daily GeoTIFFs only for now (one 1-band file for product 1, one 10-band
# file for product 2, per month) -- yearly NetCDF aggregation is a separate
# follow-up script.

# V1 - 2026-09-24 - template for schaake shuffle from insitu kriged at 1 km


# Workflow
# 1. Load config, QRF predictor bundles, spatial template
# 2. Preload the historical kriged archive ONCE per variable (the Schaake
#    reference pool) -- NOT per target month, see note below
# 3. Iterate over days in the period range
# 4. Per day:
#    a. Read that month's predictor rows from parquet
#    b. Deterministic: bundle.predict_median() -> 1-band GeoTIFF
#    c. Stochastic: bundle.sample(n=10) -> pick this day's 10 reference
#       months from the historical archive -> Schaake-shuffle -> 10-band
#       GeoTIFF
# 5. Align every output to the template extent via reproject_match, save
#    as float32 GeoTIFF

In [2]:
# libraries
import os
import gc
import glob
import joblib
import json
import numpy as np
import pandas as pd
import xarray as xr
import rioxarray as rxr
from rasterio.enums import Resampling
from scipy.stats import rankdata
from tqdm import tqdm

start_year, start_month = 2001, 1
end_year,   end_month  = 2025, 12

N_REALIZATIONS = 10
random_state_num = 64   

# paths
template = rxr.open_rasterio(r"F:\ClimateDataWS\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif")
out_base = r"F:\ClimateDataWS\Data\Processed\Monthly_downscaling"
model_path = os.path.join(out_base, r"Saved_regressors")

tmax_temp_dir = os.path.join(out_base, 'Temp_tmax')
tmax_yearly_parquet_dir = os.path.join(tmax_temp_dir, 'Tmax_yearly_parquet')
tmax_final_pred_dir = os.path.join(out_base, "QRF_Products", "Tmax")
tmax_deterministic_dir = os.path.join(tmax_final_pred_dir, "Deterministic")
tmax_stochastic_dir = os.path.join(tmax_final_pred_dir, "Stochastic")

tmin_temp_dir = os.path.join(out_base, 'Temp_tmin')
tmin_yearly_parquet_dir = os.path.join(tmin_temp_dir, 'Tmin_yearly_parquet')
tmin_final_pred_dir = os.path.join(out_base, "QRF_Products", "Tmin") 
tmin_deterministic_dir = os.path.join(tmin_final_pred_dir, "Deterministic")
tmin_stochastic_dir = os.path.join(tmin_final_pred_dir, "Stochastic")

for d in [tmax_deterministic_dir, tmax_stochastic_dir,
          tmin_deterministic_dir, tmin_stochastic_dir]:
    os.makedirs(d, exist_ok=True)


def get_model_features(t_var: str) -> list:
    """Predictor feature list straight from the fitting script's metadata"""
    metadata_file = os.path.join(model_path, f"{t_var}_qrf_feature_metadata.json")
    with open(metadata_file, 'r') as f:
        metadata = json.load(f)
    return metadata['input_features']

predictor_features_tmax = get_model_features("tmax")
predictor_features_tmin = get_model_features("tmin")

In [3]:
# HELPERS 

class QRFPredictor:
    """
    Bundle the fitted imputer + QRF pipeline with two read modes
    .predict_median(X) for median read (deterministic)
    .sample(X, n_realizations) for stochastic draws using the CDF
    """
    def __init__(self, pipeline, y_train, input_features, selected_features):
        self.pipeline = pipeline
        self.y_train = np.asarray(y_train)
        self.input_features = input_features
        self.selected_features = selected_features
    
    @property
    def qrf_(self):
        """The fitted RandomForestQuantileRegressor."""
        return self.pipeline.named_steps['reg']
    
    def transform_to_input(self, X):
        """Impute X; all columns in `input_features` pass straight to QRF."""
        return self.pipeline.named_steps['imputer'].transform(X[self.input_features])
    
    def predict_quantile(self, X, quantiles=0.5):
        return self.qrf_.predict(self.transform_to_input(X), quantiles=quantiles)
    
    def predict_median(self, X):
        return self.predict_quantile(X, quantiles=0.5)
    
    def sample(self, X, n_realizations=10, random_state=None):
        rng = np.random.default_rng(random_state)
        random_quantiles = rng.uniform(0.0, 1.0, size=n_realizations)
        return self.qrf_.predict(self.transform_to_input(X), quantiles=random_quantiles.tolist())


def load_historical_archive(t_variable: str, start_year: int, end_year: int,
                                    template) -> xr.DataArray:
    """
    Preload every available monthly kriging raster aligned to `template`.
    """
    months = pd.period_range(start=f"{start_year}-{start_month:02d}", end=f"{end_year}-{end_month:02d}", freq="M")
    arrays, valid_months = [], []
    
    # FIX: Loop over `months`, not undefined `month`
    for m in tqdm(months, desc=f"Preloading {t_variable} kriged historical archive"): 
        # FIX: Changed d.year/d.month to m.year/m.month
        path = fr"F:\ClimateDataWS\Data\Processed\Monthly_downscaling\Input_rasters\IK\ik_{t_variable}_average_{m.year}{m.month:02d}.tif"
        if not os.path.exists(path):
            continue
        
        da = rxr.open_rasterio(path, masked=True).rio.write_crs("EPSG:4326")
        da = da.squeeze("band", drop=True)
        arrays.append(da.values.astype("float32"))
        
        # FIX: Append timestamp directly so xarray doesn't trip on pd.Period
        valid_months.append(m.to_timestamp()) 
        da.close()

    archive = xr.DataArray(
        np.stack(arrays, axis=0),
        dims=("date", "y", "x"),
        coords={"date": pd.DatetimeIndex(valid_months), "y": template.y.values, "x": template.x.values},
    )
    
    # FIX: Corrected valid_dates/dates to valid_months/months
    print(f"  [{t_variable}] historical archive: {archive.shape} "
          f"({len(valid_months)}/{len(months)} months found), "
          f"{archive.nbytes / 1e9:.3f} GB in memory")
    return archive

def get_window_months(archive_months: pd.DatetimeIndex, target_month: pd.Timestamp) -> pd.DatetimeIndex:
    # FIX: Compare archive_months.month to target_month.month instead of the Timestamp object itself
    # Also removed unused window_months argument
    mask = (archive_months.month == target_month.month) & (archive_months.year != target_month.year)
    return archive_months[mask]

def select_reference_months(candidate_months: pd.DatetimeIndex, n_realizations: int,
                            rng: np.random.Generator) -> np.ndarray:
    replace = len(candidate_months) < n_realizations
    return rng.choice(candidate_months, size=n_realizations, replace=replace)

def schaake_shuffle_grid(raw_ensemble: np.ndarray, Y: np.ndarray) -> np.ndarray:
    out = raw_ensemble.copy()
    valid = ~np.isnan(Y).any(axis=1) & ~np.isnan(raw_ensemble).any(axis=1)
    if valid.any():
        x_sorted = np.sort(raw_ensemble[valid], axis=1)
        y_rank = rankdata(Y[valid], method="ordinal", axis=1).astype(int) - 1
        out[valid] = np.take_along_axis(x_sorted, y_rank, axis=1)
    return out

def build_reference_Y(archive: xr.DataArray, selected_months: np.ndarray,
                       pixel_y: np.ndarray, pixel_x: np.ndarray) -> np.ndarray:
    Y = archive.sel(date=selected_months).sel(
        y=xr.DataArray(pixel_y, dims="pixel"),
        x=xr.DataArray(pixel_x, dims="pixel"),
        method="nearest",
    )
    return Y.transpose("pixel", "date").values


def build_multiband_geotiff(df: pd.DataFrame, value_cols: list, template, outpath: str,
                             nodata: float = -9999) -> None:
    ds = df.set_index(["y", "x"])[value_cols].to_xarray()
    da = ds.to_array(dim="band")
    da = (
        da.rio.write_crs(template.rio.crs)
          .rio.write_nodata(nodata)
          .rio.write_transform(template.rio.transform())
    )
    da = da.rio.reproject_match(template)
    da.rio.to_raster(outpath, dtype="float32")

In [4]:
# Main prediction Loop

months = pd.period_range(start=f"{start_year}-{start_month:02d}", end=f"{end_year}-{end_month:02d}", freq="M")

for t_variable in ['tmin']:
    print(f"\n Predicting monthly rasters for {t_variable} ==========================================")

    predictor_features = predictor_features_tmax if t_variable == 'tmax' else predictor_features_tmin
    deterministic_dir = tmax_deterministic_dir if t_variable == 'tmax' else tmin_deterministic_dir
    stochastic_dir = tmax_stochastic_dir if t_variable == 'tmax' else tmin_stochastic_dir
    yearly_parquet_dir = tmax_yearly_parquet_dir if t_variable == 'tmax' else tmin_yearly_parquet_dir

    bundle_file = os.path.join(model_path, f"{t_variable}_qrf_predictor.pkl")
    bundle: QRFPredictor = joblib.load(bundle_file)

    raster_archive = load_historical_archive(t_variable, start_year, end_year, template)
    archive_months = raster_archive.date.to_index()

    for p in tqdm(months, desc="Predict rasters"):
        year, month = p.year, p.month
        
        # FIX: Ensure a valid timestamp is created by passing the day keyword argument
        target_month = pd.Timestamp(year=year, month=month, day=1)

        det_outpath = os.path.join(deterministic_dir, f"{t_variable}_QRFDe_{year}{month:02d}.tif")
        sto_outpath = os.path.join(stochastic_dir, f"{t_variable}_QRFSt_{year}{month:02d}.tif")

        if os.path.exists(det_outpath) and os.path.exists(sto_outpath):
            continue

        df_year_path = os.path.join(yearly_parquet_dir, fr"full_{year}.parquet")
        df_month = pd.read_parquet(
            df_year_path,
            filters=[("year", "==", year), ("month", "==", month)])

        if df_month.empty:
            tqdm.write(f"  [{year}-{month:02d}] no rows found, skipping.")
            continue

        # ---- Product 1: deterministic ----
        if not os.path.exists(det_outpath):
            df_month[f"{t_variable}_deterministic"] = bundle.predict_median(df_month[predictor_features])
            build_multiband_geotiff(df_month, [f"{t_variable}_deterministic"], template, det_outpath)

        # ---- Product 2: stochastic, 10 realizations, Schaake-shuffled ----
        if not os.path.exists(sto_outpath):
            seed_seq = np.random.SeedSequence([random_state_num, p.ordinal])
            qrf_seed, shuffle_seed = seed_seq.spawn(2)

            raw_samples = bundle.sample(
                df_month[predictor_features],
                n_realizations=N_REALIZATIONS,
                random_state=qrf_seed,
            )

            candidates = get_window_months(archive_months, target_month)
            if len(candidates) == 0:
                tqdm.write(f"  [{year}-{month:02d}] no historical candidate months in the raster archive")
                break
            else:
                rng = np.random.default_rng(shuffle_seed)
                selected_months = select_reference_months(candidates, N_REALIZATIONS, rng)
                Y = build_reference_Y(raster_archive, selected_months, df_month['y'].values, df_month['x'].values)
                shuffled = schaake_shuffle_grid(raw_samples, Y)

            realization_cols = [f"r{i:03d}" for i in range(N_REALIZATIONS)]
            df_month = pd.concat(
                [df_month, pd.DataFrame(shuffled, columns=realization_cols, index=df_month.index)],
                axis=1,
            )
            
            build_multiband_geotiff(df_month, realization_cols, template, sto_outpath)

    del raster_archive
    gc.collect()
    print(f"\nDone. All rasters saved for {t_variable}")


 Predicting monthly rasters for tmin ==========================================


Preloading tmin kriged historical archive: 100%|██████████| 300/300 [00:24<00:00, 12.48it/s]


  [tmin] historical archive: (300, 493, 983) (300/300 months found), 0.582 GB in memory


Predict rasters:  12%|█▏        | 37/300 [00:00<00:00, 321.48it/s][Parallel(n_jobs=5)]: Using backend ThreadingBackend with 5 concurrent workers.
[Parallel(n_jobs=5)]: Done  40 tasks      | elapsed:    0.1s
[Parallel(n_jobs=5)]: Done 190 tasks      | elapsed:    0.6s
[Parallel(n_jobs=5)]: Done 440 tasks      | elapsed:    1.6s
[Parallel(n_jobs=5)]: Done 790 tasks      | elapsed:    3.0s
[Parallel(n_jobs=5)]: Done 1000 out of 1000 | elapsed:    3.9s finished
Predict rasters:  14%|█▍        | 43/300 [00:58<07:47,  1.82s/it] [Parallel(n_jobs=5)]: Using backend ThreadingBackend with 5 concurrent workers.
[Parallel(n_jobs=5)]: Done  40 tasks      | elapsed:    0.2s
[Parallel(n_jobs=5)]: Done 190 tasks      | elapsed:    1.2s
[Parallel(n_jobs=5)]: Done 440 tasks      | elapsed:    2.6s
[Parallel(n_jobs=5)]: Done 790 tasks      | elapsed:    4.4s
[Parallel(n_jobs=5)]: Done 1000 out of 1000 | elapsed:    5.5s finished
[Parallel(n_jobs=5)]: Using backend ThreadingBackend with 5 concurrent worke


Done. All rasters saved for tmin
